# 01 — Engine sanity checks

Goal: verify your `Value` class is correct by computing gradients and comparing them to PyTorch.

If everything in this notebook runs and the numbers match the expected values, **Phase 1 is done**.

## Setup

In [2]:
import sys
sys.path.insert(0, '..')  # so `from nanograd...` works from the notebooks/ folder

from nanograd.engine import Value
print('nanograd imported ok')

nanograd imported ok


## Test 1 — Forward pass on a single op

Just check that the basic ops produce the right numbers.

In [3]:
a = Value(3.0)
b = Value(-2.0)

print('a + b =', (a + b).data)       # expect 1.0
print('a * b =', (a * b).data)       # expect -6.0
print('a ** 2 =', (a ** 2).data)     # expect 9.0
print('a - b =', (a - b).data)       # expect 5.0
print('a / b =', (a / b).data)       # expect -1.5
print('-a =', (-a).data)             # expect -3.0
print('2 * a =', (2 * a).data)       # expect 6.0  (uses __rmul__)
print('a.relu() =', a.relu().data)   # expect 3.0
print('b.relu() =', b.relu().data)   # expect 0.0
print('a.tanh() =', a.tanh().data)   # expect ~0.9951

a + b = 1.0
a * b = -6.0
a ** 2 = 9.0
a - b = 5.0
a / b = -1.5
-a = -3.0
2 * a = 6.0
a.relu() = 3.0
b.relu() = 0
a.tanh() = 0.9950547536867305


## Test 2 — Gradient of a single multiplication

For `c = a * b`, calculus says `dc/da = b` and `dc/db = a`.

In [4]:
a = Value(3.0)
b = Value(4.0)
c = a * b
c.backward()

print(f'a.grad = {a.grad}  (expected 4.0)')
print(f'b.grad = {b.grad}  (expected 3.0)')
assert a.grad == 4.0
assert b.grad == 3.0
print('PASS')

a.grad = 4.0  (expected 4.0)
b.grad = 3.0  (expected 3.0)
PASS


## Test 3 — Chain of ops (multi-step backward)

This exercises your topological sort. If `backward()` walks the graph correctly, the gradients should match calculus.

In [5]:
a = Value(2.0)
b = Value(3.0)
c = a * b           # 6
d = c + 5           # 11
e = d * 2           # 22
e.backward()

# e = a*b*2 + 10
# de/da = 2*b = 6
# de/db = 2*a = 4
print(f'a.grad = {a.grad}  (expected 6.0)')
print(f'b.grad = {b.grad}  (expected 4.0)')
assert a.grad == 6.0
assert b.grad == 4.0
print('PASS')

a.grad = 6.0  (expected 6.0)
b.grad = 4.0  (expected 4.0)
PASS


## Test 4 — Repeated use of the same Value

If a `Value` is used in multiple places, its gradient should ACCUMULATE (this is why we use `+=` in `_backward`).

Example: `c = a + a`. Then `dc/da = 2`.

In [6]:
a = Value(5.0)
c = a + a
c.backward()
print(f'a.grad = {a.grad}  (expected 2.0)')
assert a.grad == 2.0
print('PASS')

a.grad = 2  (expected 2.0)
PASS


## Test 5 — Karpathy's contrived expression (the big one)

This is from the README. It exercises every operator. If your gradients match, your engine is **complete and correct**.

In [7]:
a = Value(-4.0)
b = Value(2.0)
c = a + b
d = a * b + b**3
c += c + 1
c += 1 + c + (-a)
d += d * 2 + (b + a).relu()
d += 3 * d + (b - a).relu()
e = c - d
f = e**2
g = f / 2.0
g += 10.0 / f

print(f'g.data = {g.data:.4f}  (expected 24.7041)')
g.backward()
print(f'a.grad = {a.grad:.4f}  (expected 138.8338)')
print(f'b.grad = {b.grad:.4f}  (expected 645.5773)')

assert abs(g.data - 24.7041) < 1e-4
assert abs(a.grad - 138.8338) < 1e-4
assert abs(b.grad - 645.5773) < 1e-4
print('\nPHASE 1 COMPLETE')

g.data = 24.7041  (expected 24.7041)
a.grad = 138.8338  (expected 138.8338)
b.grad = 645.5773  (expected 645.5773)

PHASE 1 COMPLETE
